# Stage 3 — ESCO Occupation Classification (Batch API)
## Notebook 3.4 of 7 — Split Missing and Complete Classifications

**Role in the pipeline:**
After the first classification pass (notebooks 3.1–3.3), some vacancies may still lack an ESCO code because the LLM returned an empty or malformed response. This notebook identifies those vacancies, counts them, splits them into a separate "missing" pickle file, and removes them from the main result pickle — leaving each result file containing only successfully classified records.

The missing records are passed to notebooks 3.5–3.7 for a second classification attempt, using the approved `gpt-4o-mini` model, with the description used as fallback input.

**Position in Stage 3 sequence:**
1. 3.1 — Create batch input files
2. 3.2 — Submit batch jobs and monitor status
3. 3.3 — Extract classification results
4. ▶ **3.4 — Split missing and complete classifications** ← *you are here*
5. 3.5 — Create batch input files for missing records
6. 3.6 — Submit batch jobs for missing records *(run after Batch API completes)*
7. 3.7 — Extract results for missing records

**Prerequisites:**
- Notebook 3.3 completed — `result_status == 'complete'` for all rows

**Documentation:** [Notebook guide](README.md) · [Stage data description](../../data/data-pipeline/stage_03/README_data.md)


## 3.3.1. Load packages and configuration

Loads standard libraries and the shared `general` config module.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g
g.clean_memory()

Imports Stage 2/3 processing modules and pandas.

In [ ]:
import stage2 as st2
import stage3 as st3
import pandas as pd

## 3.3.2. Load stage 3 logs

Reads the current tracker. The commented-out lines show how to initialise the `missing_count` and `missing_path` columns on the very first run of this notebook (they don't exist in the tracker until created here).

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)
process_df

## Read missing count and split data to missing and complete

Splitting loop: iterates over all rows that have not yet been processed (skips rows already marked `missing_input_batch_status == 'created'` or `'empty'`).

For each row:
1. Loads the result pickle and counts vacancies with a missing or very short `esco_title` (NaN or length < 3).
2. Writes the `missing_count` to the tracker.
3. **If `missing > 0`:** writes missing rows to a separate pickle at `STAGE3_RESULT_PATH/missing/`, stores the path in `missing_path`, and overwrites the result pickle with only the complete (non-missing) rows.
4. **If `missing == 0`:** no split is needed; `missing_path` remains `None`.

After this step, each daily result file contains only classified records; unclassified records are in the `missing/` subfolder.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE3_PROCESS_PATH)

# Initialise all columns used by the second classification pass. This makes
# the notebook safe on a tracker produced directly by notebooks 3.1-3.3.
second_pass_columns = [
    "missing_count",
    "missing_path",
    "missing_input_batch_path",
    "missing_input_batch_status",
    "missing_job_id",
    "missing_job_status",
    "missing_output_batch_path",
    "missing_output_batch_status",
    "complete_result_status",
    "missing_after",
]
for column in second_pass_columns:
    if column not in process_df.columns:
        process_df[column] = pd.NA

for i, row in process_df.iterrows():

    if row["missing_input_batch_status"] == "created":
        continue
    if row["missing_input_batch_status"] == "empty":
        continue

    try:
        df = pd.read_pickle(row["result_path"])
    except Exception:
        print(f"Error reading file {row['result_path']}")
        continue

    total = len(df)

    if "esco_title" in df.columns:
        ser = df["esco_title"]
        missing_mask = ser.isna() | (ser.astype(str).str.len() < 3)
        missing = int(missing_mask.sum())
    else:
        missing = total  # if column absent, consider all missing

    process_df.loc[i, "missing_count"] = missing

    if missing > 0:
        missing_path = os.path.join(g.Config.STAGE3_RESULT_PATH, "missing", row["input_file"] + ".pkl")
        missing_df = df[missing_mask]
        missing_df.to_pickle(missing_path)
        process_df.loc[i, "missing_path"] = missing_path

        df = df[~missing_mask]
        df.to_pickle(process_df.loc[i, "result_path"])

    print("Working on: ", row["input_file"], " - missing: ", missing, " - total: ", total)

process_df.to_pickle(g.Config.STAGE3_PROCESS_PATH)

process_df

Displays the DataFrame of missing (unclassified) records from the last processed file — useful for inspecting what was moved to the `missing/` folder.

In [ ]:
missing_df

Displays the complete (classified) records remaining in the result DataFrame after the missing rows have been split out.

In [ ]:
df

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.